# Sentiment Analysis + Six Hypothesis Tests — AI Learning Tool A/B Test

Adapted from the course notebook to fit **ABtest Responses.xlsx** (Product A vs Product B, n = 24).

**Six hypotheses (H1–H6):** Product B is rated higher than Product A on **Accuracy, Efficiency, Knowledge, Satisfaction, Willingness to pay, Clarity** (Q4 vs Q6 matrix items, 1–7 scale).

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Upload file and read the data
Upload **ABtest Responses.xlsx** when prompted.

In [ ]:
from google.colab import files

def upload_read():
  f = files.upload()
  keys = list(f.keys())
  if keys[0].endswith('xlsx'):
    return pd.read_excel(keys[0])
  elif keys[0].endswith('csv'):
    return pd.read_csv(keys[0])
  else:
    return 'Neither Excel nor CSV file, please check the uploaded file'

data = upload_read()

In [ ]:
data.head()

In [ ]:
data.info()

# Rename columns (adapted to this survey design)
Column indices match ABtest Responses.xlsx: Q4 items (cols 4–9) are Product A ratings, Q6 items (cols 11–16) are Product B ratings, Q5/Q7 (cols 10/17) are the open comments.

In [ ]:
rename_map = {
    4:  'Accuracy_A',           11: 'Accuracy_B',
    5:  'Efficiency_A',         12: 'Efficiency_B',
    6:  'Knowledge_A',          13: 'Knowledge_B',
    7:  'Satisfaction_A',       14: 'Satisfaction_B',
    8:  'Willingness_to_pay_A', 15: 'Willingness_to_pay_B',
    9:  'Clarity_A',            16: 'Clarity_B',
    10: 'Comments_A',           17: 'Comments_B',
    18: 'Q8_noticed_difference',
    19: 'Q9_visuals_helped',
    20: 'Q10_recap_helped',
    21: 'Q11_preference',
    22: 'Q12_suggestions_B',
}
data = data.rename(columns={data.columns[i]: name for i, name in rename_map.items()})
data.columns.tolist()

# Sentiment analysis using VADER
The original `get_sentiment_score` had two bugs, fixed here:
1. `(sentence!='' or sentence!=' ' or len(sentence)==0)` was always `True` (mixed-up `or` logic), so it did nothing — replaced with a proper empty-string check that routes blanks to 'No comments'.
2. The bare `except` printed the failing sentence but appended nothing, so the output list could end up shorter than the input and misalign with the DataFrame — now it appends 'Error'.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

In [ ]:
def get_sentiment_score(all_sentences):
  """Classify each sentence as Positive / Negative / Neutral / No comments."""
  sentiment_class_list = []
  for sentence in all_sentences:
    try:
      if pd.isna(sentence) or str(sentence).strip() == '':
        sentiment_class = 'No comments'
      else:
        compound = analyzer.polarity_scores(str(sentence))['compound']
        if compound >= 0.05:
          sentiment_class = 'Positive'
        elif compound <= -0.05:
          sentiment_class = 'Negative'
        else:
          sentiment_class = 'Neutral'
      sentiment_class_list.append(sentiment_class)
    except Exception as e:
      print(f'Error on: {sentence!r} -> {e}')
      sentiment_class_list.append('Error')
  return sentiment_class_list

def get_compound_score(all_sentences):
  """Return the raw VADER compound score (NaN for empty/missing)."""
  return [np.nan if (pd.isna(s) or str(s).strip() == '')
          else analyzer.polarity_scores(str(s))['compound'] for s in all_sentences]

This survey has **one open comment per product** (Q5 for A, Q7 for B) instead of one per dimension, so sentiment is scored on those two columns (plus the Q12 suggestions).

In [ ]:
for col in ['Comments_A', 'Comments_B', 'Q12_suggestions_B']:
  data[col + '_sentiment'] = get_sentiment_score(data[col].values)
  data[col + '_compound'] = get_compound_score(data[col].values)

print('Product A comments:'); print(data['Comments_A_sentiment'].value_counts())
print('\nProduct B comments:'); print(data['Comments_B_sentiment'].value_counts())
print(f"\nMean compound: A = {data['Comments_A_compound'].mean():.3f}, B = {data['Comments_B_compound'].mean():.3f}")

In [ ]:
# Is B's comment sentiment significantly more positive than A's? (paired, same respondents)
mask = data['Comments_A_compound'].notna() & data['Comments_B_compound'].notna()
t, p = stats.ttest_rel(data.loc[mask, 'Comments_B_compound'], data.loc[mask, 'Comments_A_compound'])
w = stats.wilcoxon(data.loc[mask, 'Comments_B_compound'], data.loc[mask, 'Comments_A_compound'])
print(f'Paired t-test on compound sentiment (B vs A): t = {t:.3f}, p = {p:.4f}')
print(f'Wilcoxon signed-rank: W = {w.statistic:.1f}, p = {w.pvalue:.4f}')

# Six hypothesis tests (H1–H6)
Each respondent rated both products, so we use **paired** tests: paired t-test plus Wilcoxon signed-rank (robust check for 1–7 Likert data). One-sided p tests the directional claim B > A.

In [ ]:
dims = [
    ('H1', 'Accuracy',           'Product B is rated more accurate than Product A'),
    ('H2', 'Efficiency',         'Product B helps users understand more quickly than Product A'),
    ('H3', 'Knowledge',          'Product B is rated more knowledgeable than Product A'),
    ('H4', 'Satisfaction',       'Users are more satisfied with Product B than Product A'),
    ('H5', 'Willingness_to_pay', 'Users are more willing to pay for Product B than Product A'),
    ('H6', 'Clarity',            'Product B explanations are clearer than Product A'),
]
rows = []
for h, dim, desc in dims:
  a = data[dim + '_A'].astype(float)
  b = data[dim + '_B'].astype(float)
  diff = b - a
  t, p2 = stats.ttest_rel(b, a)
  p1 = p2 / 2 if t > 0 else 1 - p2 / 2          # one-sided (B > A)
  wp = stats.wilcoxon(b, a).pvalue
  d = diff.mean() / diff.std(ddof=1)             # Cohen's d for paired samples
  rows.append({'Hypothesis': h, 'Dimension': dim, 'Statement': desc,
               'Mean_A': round(a.mean(), 2), 'Mean_B': round(b.mean(), 2),
               'Mean_diff': round(diff.mean(), 2), 't': round(t, 3),
               'p_two_sided': round(p2, 4), 'p_one_sided_B_gt_A': round(p1, 4),
               'Wilcoxon_p': round(wp, 4), 'Cohens_d': round(d, 2),
               'Supported': 'YES' if p1 < 0.05 else 'NO'})
results = pd.DataFrame(rows)
results

# Supporting evidence (Q8–Q11)

In [ ]:
print(f"Q9  visual diagrams helped (1-7): mean = {data['Q9_visuals_helped'].mean():.2f}")
print(f"Q10 recap/practice helped  (1-7): mean = {data['Q10_recap_helped'].mean():.2f}")
print('\nQ11 Which would you keep using?'); print(data['Q11_preference'].value_counts())
print('\nQ8 Noticed differences?'); print(data['Q8_noticed_difference'].value_counts())

## Save the results into csv files

In [ ]:
results.to_csv('hypothesis_results.csv', index=False)
sent_cols = ['Comments_A', 'Comments_A_sentiment', 'Comments_A_compound',
             'Comments_B', 'Comments_B_sentiment', 'Comments_B_compound',
             'Q12_suggestions_B', 'Q12_suggestions_B_sentiment']
data[sent_cols].to_csv('survey_with_sentiments.csv', index=False)
files.download('hypothesis_results.csv')
files.download('survey_with_sentiments.csv')